<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>DIFFERENTIABLE AUTOENCODING NEURAL OPERATOR -  TEMPORAL MARCHING</strong><br>
    <strong></strong> SIVA VIKNESH <br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from scipy.linalg import circulant, toeplitz, inv
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.animation as animation

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor, '\n')

In [ ]:
seed = 10
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
input_var    = 3                                        # INPUT VARIABLES {a(x, y), x, y}
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
neurons      = 128
epochs       = 60
batchsize    = 30 
in_channels  = 3                       # NO. OF INPUT CHANNELS
mid_channels = 32                      # WIDTH OF THE CNN LAYERS IN THE SPECTRAL CONVOLUTION
out_channels = 1                       # NO. OF OUTPUT CHANNELS

learn_rate   = 1e-2
step_epoch   = 5
decay_rate   = 0.750

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_256_0.vtk"
data_file    = "Grid_256_"
Nfile        = 1000                                      # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../")
directory    = os.getcwd()
path         = directory + "/Flow_Data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale   = 10.0
yscale   = 5.0
vort_max = 4.4

Nbn      = 16
dt       = 0.01
mu       = 0.01
velocity = 1
h        = 1/Nbn

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))/yscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname1  = 'u_vel'                                                      # FIELD NAME FOR VTK FILES
fieldname2  = 'v_vel'
fieldname3  = 'vort_z'

#u_vel   = np.zeros((Nfile, x_dim, y_dim))
#v_vel   = np.zeros((Nfile, x_dim, y_dim))
vort    = np.zeros((Nfile, x_dim, y_dim))

for i in range(Nfile):
    file_name = data_file + str(i) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput()
    #pointData1 = data.GetPointData().GetArray(fieldname1)
    #pointData2 = data.GetPointData().GetArray(fieldname2)    
    pointData3 = data.GetPointData().GetArray(fieldname3)    
    #u_vel   [i, :, :] = np.reshape(vtk_to_numpy(pointData1), (x_dim, y_dim))
    #v_vel   [i, :, :] = np.reshape(vtk_to_numpy(pointData2), (x_dim, y_dim))
    vort    [i, :, :] = np.reshape(vtk_to_numpy(pointData3), (x_dim, y_dim))
    
    print ("*"*85)
vort = vort/vort_max

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE ARCHITECTURE


In [ ]:
def COMPLEX_MULTIPLY(input_data, weights):                             # COMPLEX WEIGHTS X FFT MODES 

    return torch.einsum("bixy,ioxy->boxy", input_data, weights)

class CNN_LAYERS (nn.Module):                                           # LIFITNG/ PROJECTING THE INPUT LAYERS

    def __init__(self, in_channel, mid_channel, out_channel):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels = in_channel, out_channels= mid_channel, kernel_size=1),
            nn.GELU(),
            
            nn.Conv2d(in_channels = mid_channel, out_channels = out_channel, kernel_size=1)
            #nn.GELU(),

            #nn.Conv2d(in_channels = mid_channel, out_channels = out_channel, kernel_size=1)

            # MAY INCLUDE DROPOUT LAYERS nn.Dropout(p=0.5)
            )

    def forward (self, x):
        output = self.main(x)        
        return output

class SPECTRAL_BLOCK (nn.Module):
    def __init__(self, in_channels, out_channels, x_modes, y_modes):
        super().__init__()    

        # INITIALISATION OF COMPLEX WEIGHT FUNCTIONS 
        self.out_channels = out_channels 
        self.in_channels  = in_channels 
        self.scale        =  1 / (self.in_channels + self.out_channels)
        self.modes1       = x_modes      #Number of Fourier modes to multiply, at most floor(N/2) + 1
        self.modes2       = y_modes
        self.x_weights    = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))
        self.y_weights    = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def forward (self, x):

        batchsize = x.shape[0]

        # TRANSFORM THE GIVEN BATCH TO A FOURIER SPACE - 2D REAL FFT COMPUTATION
        x_fft = torch.fft.rfft2(x)

        # RETANING ONLY THE DESIRED FOURIER MODES
        out_fft = torch.zeros(batchsize, self.out_channels,  x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat).to(processor)

        out_fft[:, :, :self.modes1, :self.modes2]  = COMPLEX_MULTIPLY (x_fft[:, :, :self.modes1, :self.modes2], self.x_weights)

        out_fft[:, :, -self.modes1:, :self.modes1] = COMPLEX_MULTIPLY (x_fft[:, :, -self.modes1:, :self.modes2], self.y_weights)

        # TRANSFORM THE GIVEN BATCH BACK TO THE PHYSICAL SPACE - 2D REAL IFFT COMPUTATION
        x = torch.fft.irfft2(out_fft, s=(x.size(-2), x.size(-1)))

        return x


class ENCODER (nn.Module):

    def __init__(self, input_var, x_modes, y_modes, width, x_grid, y_grid):
        super().__init__()
        self.x_grid   = x_grid
        self.y_grid   = y_grid
        self.x_modes  = x_modes
        self.y_modes  = y_modes
        self.width    = width
        #self.padding  = 9                                          # PAD THE DOMAIN IF THE INPUT IS NON-PERIODIC

        self.p        = nn.Linear (3, self.width)                   # INPUT CHANNEL IS 3: (a(x, y), x, y)
        
        self.conv0    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv1    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv2    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv3    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)

        self.mlp0     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp1     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp2     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp3     = CNN_LAYERS (self.width, self.width, self.width)

        self.w0       = nn.Conv2d  (self.width, self.width, 1)
        self.w1       = nn.Conv2d  (self.width, self.width, 1)
        self.w2       = nn.Conv2d  (self.width, self.width, 1)
        self.w3       = nn.Conv2d  (self.width, self.width, 1)

        self.q        = CNN_LAYERS (self.width, self.width * 4, 1)  # OUTPUT CHANNEL IS 1: u(x, y)
        self.reduce   = nn.AvgPool2d(1, stride = 2)

    def forward (self, x):
        batch = x.shape[0]
        xgrid = torch.Tensor(self.x_grid).reshape(1, x_dim, y_dim, 1).repeat(x.shape[0], 1, 1, 1)
        ygrid = torch.Tensor(self.y_grid).reshape(1, x_dim, y_dim, 1).repeat(x.shape[0], 1, 1, 1)
        grid = torch.cat((xgrid, ygrid), dim=-1).to(processor)
        x = torch.cat((x, grid), dim=-1)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)

        #x = F.pad(x, [0,self.padding, 0,self.padding])
        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))
        
        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))
    
        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)
        x = self.reduce (torch.squeeze(x))

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2
        x = self.reduce (torch.squeeze(x))

        #x = x[..., :-self.padding, :-self.padding]
        x = self.q(x)
        #x = x.permute(0, 2, 3, 1)
        return x

class DECODER (nn.Module):

    def __init__(self, x_modes, y_modes, width):
        super().__init__()

        self.x_modes  = x_modes
        self.y_modes  = y_modes
        self.width    = width
        self.padding  = 9                                           # PAD THE DOMAIN IF THE INPUT IS NON-PERIODIC

        self.p        = nn.Linear (1, self.width)                   # INPUT CHANNEL IS 3: (a(x, y), x, y)

        self.conv0    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv1    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv2    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)
        self.conv3    = SPECTRAL_BLOCK (self.width, self.width, self.x_modes, self.y_modes)

        self.mlp0     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp1     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp2     = CNN_LAYERS (self.width, self.width, self.width)
        self.mlp3     = CNN_LAYERS (self.width, self.width, self.width)

        self.w0       = nn.Conv2d  (self.width, self.width, 1)
        self.w1       = nn.Conv2d  (self.width, self.width, 1)
        self.w2       = nn.Conv2d  (self.width, self.width, 1)
        self.w3       = nn.Conv2d  (self.width, self.width, 1)

        self.q        = CNN_LAYERS (self.width, self.width * 4, 1)  # OUTPUT CHANNEL IS 1: u(x, y)
        self.upsamp   = nn.ConvTranspose2d(self.width, self.width, 2, stride = 2)

    def forward (self, x):
        #x = F.pad(x, [0,self.padding, 0,self.padding])
        x  = x.permute(0, 2, 3, 1)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)
        x  = self.upsamp(x)
        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x  = x1 + x2
        x  = F.gelu(x)

        x  = self.upsamp (x)
        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x  = x1 + x2
        x  = F.gelu(x)

        x  = self.upsamp (x)
        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x  = x1 + x2
        x  = F.gelu(x)

        x  = self.upsamp (x)
        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x  = x1 + x2
        
        #x  = self.upsamp (x)
        #x  = x[..., :-self.padding, :-self.padding]
        x  = self.q(x)
        x  = x.permute(0, 2, 3, 1)

        return x

In [ ]:
class RK4_METHOD(nn.Module):
    def __init__(self, dt):
        super().__init__()
        self.dt = dt

    def forward(self, RHS, x):
        k1 = RHS                 
        k2 = RHS + 0.5 * self.dt * k1
        k3 = RHS + 0.5 * self.dt * k2
        k4 = RHS + self.dt * k3
        x = x + self.dt * (k1 / 6 + k2 / 3 + k3 / 3 + k4 / 6)
        return x
    
class CD2(nn.Module):  # SECOND DERIVATIVE
    def __init__(self, Nx, Ny, hx, hy):
        super().__init__()
        self.Nx = Nx
        self.hx = hx
        self.Ny = Ny
        self.hy = hy
        # Register buffers for the second derivative matrices
        self.register_buffer("Dxx", self.compute_cds_matrix(Nx, hx))
        self.register_buffer("Dyy", self.compute_cds_matrix(Ny, hy))

    def compute_cds_matrix(self, N, h):
        a11, a12, a13 = 1.0, -2.0, 1.0
        d = np.zeros(N)
        d[-1], d[0], d[1] = a11, a12, a13
        
        # Construct the circulant matrix for the second derivative operator
        D = torch.tensor(circulant(d), dtype=torch.float32, device=processor) / (h**2)

        # Apply boundary conditions
        D[0, :] = 0.0
        D[0, :4] = torch.tensor([2.0, -5.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (h**2)
        D[-1, :] = 0.0
        D[-1, -4:] = torch.tensor([-1.0, 4.0, -5.0, 2.0], dtype=torch.float32, device=processor) / (h**2)

        return D

    def forward(self, x):
        # Compute the second derivatives using matrix multiplication
        d2dxx = torch.einsum('ij, jk -> ik', self.Dxx, x)
        d2dyy = torch.einsum('ij, kj -> ki', self.Dyy, x)
        return d2dxx, d2dyy
    
class CDS(nn.Module):  # FIRST DERIVATIVE
    def __init__(self, Nx, Ny, hx, hy):
        super().__init__()
        self.Nx = Nx
        self.hx = hx
        self.Ny = Ny
        self.hy = hy
        # Register buffers for the first derivative matrices
        self.register_buffer("Dx", self.compute_cds_matrix(Nx, hx))
        self.register_buffer("Dy", self.compute_cds_matrix(Ny, hy))

    def compute_cds_matrix(self, N, h):
        a11, a12, a13 = 1.0, 0.0, -1.0
        d = np.zeros(N)
        d[-1], d[0], d[1] = a11, a12, a13

        # Use scipy's circulant function to generate the first derivative matrix
        D = torch.tensor(circulant(d), dtype=torch.float32, device=processor)/ (2.0 * h)

        # Apply boundary conditions
        D[0, :] = 0.0
        D[0, :3] = torch.tensor([-3.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (2.0 * h)
        D[-1, :] = 0.0
        D[-1, -3:] = torch.tensor([1.0, -4.0, 3.0], dtype=torch.float32, device=processor) / (2.0 * h)

        return D

    def forward(self, x):
        # Compute the first derivatives using matrix multiplication
        ddx = torch.einsum('ij, jk -> ik', self.Dx, x)
        ddy = torch.einsum('ij, kj -> ki', self.Dy, x)
        return ddx, ddy
    
class OUCS2(nn.Module):
    def __init__(self, Nx, Ny, hx, hy):
        super().__init__()
        self.Nx = Nx
        self.Ny = Ny
        self.hx = hx
        self.hy = hy
        
        # Register buffers as circulant matrices
        self.register_buffer("Dx", self.compute_OUCS_matrix(Nx, hx))
        self.register_buffer("Dy", self.compute_OUCS_matrix(Ny, hy))

    def compute_OUCS_matrix(self, N, h):
        d = np.zeros(N)
        
        a = -40.0
        p33 = 36.0
        p32 = p33 / 3.0 - a / 12.0
        p34 = p33 / 3.0 + a / 12.0

        d[1] = p32
        d[0] = p33
        d[-1] = p34
        
        D1 = torch.tensor(circulant(d), dtype=torch.float32, device=processor)
        
        d = np.zeros(N)
        
        q31 = -p33 / 36.0 + a / 72.0
        q32 = -7.0 * p33 / 9.0 + a / 9.0
        q33 = -a / 4.0
        q34 = 7.0 * p33 / 9.0 + a / 9.0
        q35 = p33 / 36.0 + a / 72.0

        beta1 = 0.020
        beta2 = 0.090

        d[-2] = q35
        d[-1] = q34
        d[0] = q33
        d[1] = q32
        d[2] = q31

        D2 = torch.tensor(circulant(d), dtype=torch.float32, device=processor) / h

        out = torch.matmul(torch.linalg.inv(D1), D2)

        # Modify specific sections of the output matrix
        out[0:2, :] = 0.0
        out[0, :3] = torch.tensor([-3.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (2.0 * h)
        out[1, :5] = torch.tensor([((2.0 * beta1) / 3.0 - 1.0 / 3.0), -((8.0 * beta1) / 3.0 + 0.50), 4.0 * beta1 + 1.0, -(8.0 * beta1 / 3.0 + 1.0 / 6.0), 2.0 * beta1 / 3.0])

        out[-2:, :] = 0.0
        out[-2, -5:] = torch.tensor([(-2.0 * beta2 / 3.0, (8.0 * beta2 / 3.0 + 1.0 / 6.0), -(4.0 * beta2 + 1.0), ((8.0 * beta2) / 3.0 + 0.50), -((2.0 * beta2) / 3.0 - 1.0 / 3.0))])
        out[-1, -3:] = torch.tensor([1.0, -4.0, 3.0], dtype=torch.float32, device=processor) / (2.0 * h)

        return out

    def forward(self, x):
        ddx = torch.einsum('ij, jk -> ik', self.Dx, x)
        ddy = torch.einsum('ij, kj -> ki', self.Dy, x)
        return ddx, ddy

class OUCS3(nn.Module):
    def __init__(self, Nx, Ny, hx, hy):
        super().__init__()
        self.Nx = Nx
        self.Ny = Ny
        self.hx = hx
        self.hy = hy
        
        # Register buffers as circulant matrices
        self.register_buffer("Dx", self.compute_OUCS_matrix(Nx, hx))
        self.register_buffer("Dy", self.compute_OUCS_matrix(Ny, hy))

    def compute_OUCS_matrix(self, N, h):
        
        d = np.zeros(N)
        D = 0.3793894912
        E = 0.183205192
        F = 1.57557379
        G = -2.0

        p32 = D - G / 60.0
        p33 = 1.0
        p34 = D + G / 60.0

        d[1]  = p32
        d[0]  = p33
        d[-1] = p34
        
        D1 = torch.tensor(circulant(d), dtype=torch.float32, device=processor)

        d = np.zeros(N)

        q31 = -F / 4.0 + G / 300.0
        q32 = -E / 2.0 + G / 30.0
        q33 = -11.0 * G / 150.0
        q34 = E / 2.0 + G / 30.0
        q35 = F / 4.0 + G / 300.0

        beta1 = -0.025
        beta2 = 0.090

        d[-2]  = q35
        d[-1]  = q34
        d[0]   = q33
        d[1]   = q32
        d[2]   = q31

        D2 = torch.tensor(circulant(d), dtype=torch.float32, device=processor) / h

        out = torch.matmul(torch.linalg.inv(D1), D2)

        # Modify specific sections of the output matrix
        out[0:2, :] = 0.0
        out[0, :3] = torch.tensor([-3.0, 4.0, -1.0], dtype=torch.float32, device=processor) / (2.0 * h)
        out[1, :5] = torch.tensor([((2.0 * beta1) / 3.0 - 1.0 / 3.0), 
                                   -((8.0 * beta1) / 3.0 + 0.50), 
                                   4.0 * beta1 + 1.0, 
                                   -(8.0 * beta1 / 3.0 + 1.0 / 6.0), 
                                   2.0 * beta1 / 3.0], device=processor) / h

        out[-2:, :] = 0.0
        out[-2, -5:] = torch.tensor([(-2.0 * beta2 / 3.0,  
                                      (8.0 * beta2 / 3.0 + 1.0 / 6.0), 
                                      -(4.0 * beta2 + 1.0), 
                                      ((8.0 * beta2) / 3.0 + 0.50),  
                                      -((2.0 * beta2) / 3.0 - 1.0 / 3.0))], device=processor) / h
        out[-1, -3:] = torch.tensor([1.0, -4.0, 3.0], dtype=torch.float32, device=processor) / (2.0 * h)

        return out

    def forward(self, x):
        ddx = torch.einsum('ij, jk -> ik', self.Dx, x)
        ddy = torch.einsum('ij, kj -> ki', self.Dy, x)
        return ddx, ddy

    
class VORTICITY_EQN (nn.Module):
    def __init__(self, N, h, dt, mu, velocity):
        super().__init__()
        
        self.N = N
        self.h = h
        self.dt = dt
        self.mu = mu
        self.vel = velocity  
        
        self.Dx        = OUCS2 (self.N, self.N, self.h, self.h)
        self.Dxx       = CD2   (self.N, self.N, self.h, self.h)
        self.Integrate = RK4_METHOD(self.dt)
        
    def forward(self, x):
        for i in range(x.shape[0]):  # x.shape[0] = batch size
            Dxx, Dyy      = self.Dxx(torch.squeeze(x[i, 0, :, :]))  
            Dx, Dy        = self.Dx(torch.squeeze(x[i, 0, :, :]))   
            RHS           = -self.vel * (Dx + Dy) + self.mu * (Dxx + Dyy)
            x[i, 0, :, :] = self.Integrate(RHS, x[i, 0, :, :])
        return x
    
class NSE_EQUATION(nn.Module):                    # ZERO PRESSURE GRADIENT NAVIER-STOKES EQUATION
    def __init__(self, N, h, dt, mu, velocity):
        super().__init__()
        
        self.N   = N
        self.h   = h
        self.dt  = dt
        self.mu  = mu
        self.vel = velocity

        self.Dx        = OUCS2 (self.N, self.N, self.h, self.h)
        self.Dxx       = CD2   (self.N, self.N, self.h, self.h)
        self.Integrate = RK4_METHOD(self.dt)
        
    def forward(self, Vx, Vy):
        x   = torch.zeros_like (Vx)
        y   = torch.zeros_like (Vy)
        
        for i in range(Vx.shape[0]):  # Vx.shape[0] = batch size    
            # Vx component Velocity
            dudxx, dudyy   = self.Dxx (torch.squeeze(Vx[i, 0, :, :]))  
            dudx, dudy     = self.Dx  (torch.squeeze(Vx[i, 0, :, :]))
            dvdxx, dvdyy   = self.Dxx (torch.squeeze(Vy[i, 0, :, :]))  
            dvdx, dvdy     = self.Dx  (torch.squeeze(Vy[i, 0, :, :])) 
            
            dudt_RHS       = -self.vel*(dudx + dudy) + self.mu*(dudxx + dudyy)
            dvdt_RHS       = -self.vel*(dvdx + dvdy) + self.mu*(dvdxx + dvdyy)
            
            x[i, 0, :, :]  = self.Integrate(dudt_RHS, torch.squeeze(Vx[i, 0, :, :]))
            y[i, 0, :, :]  = self.Integrate(dvdt_RHS, torch.squeeze(Vy[i, 0, :, :]))
        return x, y

In [ ]:
# MESH SPATIAL DATA
x_grid = torch.Tensor(x).reshape(1, x_dim, y_dim, 1)
y_grid = torch.Tensor(y).reshape(1, x_dim, y_dim, 1)
    
# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort[:-1, :, :]).to(processor)
vort_output  = torch.Tensor(vort[ 1:, :, :]).to(processor)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

vort_input_reshaped  = vort_input.reshape  (total_samples, x_dim, y_dim, 1)
vort_output_reshaped = vort_output.reshape (total_samples, x_dim, y_dim, 1)

dataset               = TensorDataset(vort_input_reshaped, vort_output_reshaped)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10, shuffle=False)
plot_loader    = DataLoader(dataset, batch_size=batchsize, shuffle=False)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DIANO TRAINING
</div>


In [ ]:
x_modes      = 16                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)
y_modes      = 16                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)

vort_encoder = ENCODER  (input_var, x_modes, y_modes, mid_channels, x_grid, y_grid).to(processor)
diff_eqn     = VORTICITY_EQN (Nbn, h, dt, mu, velocity).to(processor)
vort_decoder = DECODER  (x_modes, y_modes, mid_channels).to(processor)

encoder_optim  = optim.Adam(vort_encoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)
decoder_optim  = optim.Adam(vort_decoder.parameters(), lr=learn_rate, betas = (0.9,0.99), eps = 10**-15)

encoder_scheduler  = optim.lr_scheduler.StepLR(encoder_optim, step_size=step_epoch, gamma=decay_rate)
decoder_scheduler  = optim.lr_scheduler.StepLR(decoder_optim, step_size=step_epoch, gamma=decay_rate)

Loss_Data   = torch.empty(size=(epochs+1, 1))
loss_func   = nn.MSELoss()

for epoch in range (epochs+1):
    total_loss = 0.0
    for batch_idx, (w_in, w_out) in enumerate(train_loader):
        encoder_output  = vort_encoder (w_in)
        march_output    = diff_eqn     (encoder_output)
        decoder_output  = vort_decoder (march_output)
        batch_loss      = loss_func    (decoder_output, w_out) 

        encoder_optim.zero_grad()
        decoder_optim.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            encoder_optim.step()
            decoder_optim.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', encoder_optim.param_groups[0]['lr'])
    print ("*"*85)
    
    encoder_scheduler.step()
    decoder_scheduler.step()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(vort_encoder.state_dict(), "VORT_ENCODER_DT.pt" )
torch.save(vort_decoder.state_dict(), "VORT_DECODER_DT.pt" )

torch.save(Loss_Data  [0:epoch], "Loss_DT.pt"  )

In [ ]:
x_modes      = 16                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)
y_modes      = 16                      # NO. OF FOURIER MODES ALONG EACH DIMENSION (EVEN INTEGER TUPLE)
vort_encoder = ENCODER  (input_var, x_modes, y_modes, mid_channels, x_grid, y_grid).to(processor)
diff_eqn     = VORTICITY_EQN (Nbn, h, dt, mu, velocity).to(processor)
vort_decoder = DECODER  (x_modes, y_modes, mid_channels).to(processor)

os.chdir(pwd)
vort_encoder.load_state_dict(torch.load("VORT_ENCODER_DT.pt", weights_only=True))
vort_decoder.load_state_dict(torch.load("VORT_DECODER_DT.pt", weights_only=True))
loss_func   = nn.MSELoss()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        TESTING DATA
</div>


In [ ]:
# TEST ERROR OF FNO
vort_encoder.eval()
vort_decoder.eval()

total_loss = 0.0
for batch_idx, (w_in, w_out) in enumerate(test_loader):
    encoder_output  = vort_encoder (w_in)
    march_output    = diff_eqn     (encoder_output)
    decoder_output  = vort_decoder (march_output)
    batch_loss      = loss_func    (decoder_output, w_out) 

    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())

In [ ]:
test_loader    = DataLoader(vort_test,  batch_size=2, shuffle=False)
for batch_idx, (w_in, w_out) in enumerate(test_loader):
    encoder_output  = vort_encoder (w_in)
    march_output    = diff_eqn     (encoder_output)
    decoder_output  = vort_decoder (march_output)
    vort_error      = torch.abs    (decoder_output- w_out)

fig, ax = plt.subplots(1, 5, figsize=(30, 10))

# Define common settings
cbar_fontsize = 20  # Increase colorbar font size
num_ticks = 5  # Reduce number of labels in the colorbar

def add_colorbar(pcm, ax):
    cbar = fig.colorbar(pcm, ax=ax, orientation='horizontal', pad=0.02)
    cbar.ax.tick_params(labelsize=cbar_fontsize)  # Increase font size
    # Reduce the number of ticks
    cbar.set_ticks(np.linspace(pcm.get_clim()[0], pcm.get_clim()[1], num_ticks))

# Row 0: Vx Plots
pcm = ax[0].contourf(torch.squeeze(w_in[0, :, :, :]).detach().cpu().numpy(), cmap='jet') 
ax[0].set_title('Input Data', fontsize=24)  
ax[0].set_xticks([]) 
ax[0].set_yticks([])
ax[0].set_box_aspect(aspect=1)
add_colorbar(pcm, ax[0])

pcm = ax[1].contourf(torch.squeeze(encoder_output[0, :, :, :]).detach().cpu().numpy(), cmap='jet') 
ax[1].set_title('Bottle Neck - Vx', fontsize=24)  
ax[1].set_xticks([]) 
ax[1].set_yticks([])
ax[1].set_box_aspect(aspect=1)
add_colorbar(pcm, ax[1])

pcm = ax[2].contourf(torch.squeeze(march_output[0, :, :, :]).detach().cpu().numpy(), cmap='jet') 
ax[2].set_title('Bottle Neck - Marched', fontsize=24)  
ax[2].set_xticks([]) 
ax[2].set_yticks([])
ax[2].set_box_aspect(aspect=1)
add_colorbar(pcm, ax[2])

pcm = ax[3].contourf(torch.squeeze(decoder_output[0, :, :, :]).detach().cpu().numpy(), cmap='jet') 
ax[3].set_title('Output Data', fontsize=24)  
ax[3].set_xticks([]) 
ax[3].set_yticks([])
ax[3].set_box_aspect(aspect=1)
add_colorbar(pcm, ax[3])

pcm = ax[4].contourf(torch.squeeze(vort_error[0, :, :, :]).detach().cpu().numpy(), cmap='jet') 
ax[4].set_title('Absolute Error', fontsize=24)  
ax[4].set_xticks([]) 
ax[4].set_yticks([])
ax[4].set_box_aspect(aspect=1)
add_colorbar(pcm, ax[4])

# Adjust layout
plt.tight_layout()

# Optionally save the figure
# plt.savefig("Fig.png")
plt.show()

In [ ]:
def save_to_tecplot(filename, tensor, title, varname="Value"):
    """
    Save a 2D tensor (torch.Tensor) to a Tecplot-compatible ASCII .dat file.
    """
    # Convert to NumPy and squeeze
    np_data = torch.squeeze(tensor).detach().cpu().numpy()
    
    # Get dimensions
    ny, nx = np_data.shape
    
    # Generate coordinate mesh (assuming uniform grid in [0, 1] x [0, 1])
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)

    # Flatten everything
    X_flat = X.flatten()
    Y_flat = Y.flatten()
    val_flat = np_data.flatten()
    
    # Write to file
    with open(filename, 'w') as f:
        f.write(f'TITLE = "{title}"\n')
        f.write(f'VARIABLES = "X", "Y", "{varname}"\n')
        f.write(f'ZONE T="{title}", I={nx}, J={ny}, F=POINT\n')
        for i in range(len(X_flat)):
            f.write(f"{X_flat[i]:.6f} {Y_flat[i]:.6f} {val_flat[i]:.6e}\n")

# Example usage
save_to_tecplot("input_velocity.dat",     w_in[0],           "Input Velocity",      "U_in")
save_to_tecplot("encoded_bottleneck.dat", encoder_output[0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("marched_data.dat",       encoder_output[0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("decoded_output.dat",     decoder_output[0], "Output Velocity",    "U_out")
save_to_tecplot("output_velocity.dat",    w_out[0],          "Output Velocity",    "U_out")
save_to_tecplot("absolute_error.dat",     vort_error[0],     "Absolute Error",    "AbsErr")

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        MAKING THE SNAPSHOTS VIDEO
</div>

In [ ]:
del train_loader, test_loader
vort_input_list   = []
vort_encoder_list = []
vort_marched_list = []
vort_decoder_list = []
vort_err_list     = []

for w_in, w_out in plot_loader:
    encoder_output  = vort_encoder (w_in)
    march_output    = diff_eqn     (encoder_output)
    decoder_output  = vort_decoder (march_output)
    vort_error      = torch.abs    (decoder_output- w_out)
    
    vort_input_list.append(w_in.detach().cpu())
    vort_encoder_list.append(encoder_output.detach().cpu())
    vort_marched_list.append(march_output.detach().cpu())
    vort_decoder_list.append(decoder_output.detach().cpu())
    vort_err_list.append(vort_error.detach().cpu())

input_data_np   = torch.cat(vort_input_list,   dim=0).squeeze().numpy()
encoder_data_np = torch.cat(vort_encoder_list, dim=0).squeeze().numpy()
marched_data_np = torch.cat(vort_marched_list, dim=0).squeeze().numpy()
decoder_data_np = torch.cat(vort_decoder_list, dim=0).squeeze().numpy()
abs_err_data_np = torch.cat(vort_err_list    , dim=0).squeeze().numpy()

# -------------------------------------------------------------------------------------------------------------------- #

def create_animation(data, filename, cmap_name='Wistia', levels=50):
    fig, ax = plt.subplots(figsize=(12, 8))
    plt.subplots_adjust(left=0.02, right=0.95, top=0.90, bottom=0.02)  # Adjust margins
    cmap = plt.get_cmap(cmap_name)

    # Initialize the contour plot
    contour = ax.contourf(data[0, :, :], cmap=cmap)
    ax.set_xticks([])  # Hide x-axis ticks
    ax.set_yticks([])  # Hide y-axis ticks

    def animate_frame(i):
        ax.clear()
        ax.contourf(data[i, :, :], cmap=cmap)
        ax.set_title(f'Time = {i * dt:.4f}', fontsize=38)
        ax.set_xticks([])
        ax.set_yticks([])
        return ax,

    ani = animation.FuncAnimation(fig, animate_frame, frames=Nfile-1, blit=False)
    FFwriter = animation.FFMpegWriter(bitrate=4096, fps=24)
    ani.save(filename, writer=FFwriter, dpi=600)
    plt.close()

# -------------------------------------------------------------------------------------------------------------------- #
# Generate animations for each dataset
create_animation(input_data_np,   'Input_data.mp4',     cmap_name='jet')
create_animation(encoder_data_np, 'Encoder_data.mp4',   cmap_name='jet')
create_animation(marched_data_np, 'Marched_data.mp4',   cmap_name='jet')
create_animation(decoder_data_np, 'Decoder_data.mp4',   cmap_name='jet')
create_animation(abs_err_data_np, 'Abs_Error_data.mp4', cmap_name='jet')
